### Tool Calling
Tool calling is method that allows an LLM to use external tools or systems to perform tasks it cannot do by itself.

In [1]:
import ollama

In [2]:
inventory_db = {
    "laptop": {"stock": 5, "base_price": 1200},
    "monitor": {"stock": 0, "base_price": 300},
    "keyboard": {"stock": 25, "base_price": 80},
}

In [3]:
# Defiine the tool function
# TOOL1: Check the database
def check_inventory(product_name):
    product = inventory_db.get(product_name)
    if not product:
        return f"Product '{product_name}' not found in inventory."
    
    stock = product["stock"]
    base_price = product["base_price"]
    
    return {"stock": stock, "base_price": base_price}

# TOOL2: Business logic for Discounts
def claculate_loyalty_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.30)  # Max discount of 30%
    final_price = base_price * (1 - discount)
    return round(final_price, 2)

In [4]:
# 1. Map the function so the script can call them by name
available_functions = {
    "check_inventory": check_inventory,
    "claculate_loyalty_discount": claculate_loyalty_discount,
}

In [5]:
# Define schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check the stock and base price of a product.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string", "description": "Name of the product to check."}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "claculate_loyalty_discount",
            "description": "Calculate the loyalty discount for a customer based on their years of service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number", "description": "The base price of the product."},
                    "years_as_customer": {"type": "integer", "description": "The number of years the customer has been with the company."}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [6]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Check the stock and base price of a product.',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string',
      'description': 'Name of the product to check.'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'claculate_loyalty_discount',
   'description': 'Calculate the loyalty discount for a customer based on their years of service.',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number',
      'description': 'The base price of the product.'},
     'years_as_customer': {'type': 'integer',
      'description': 'The number of years the customer has been with the company.'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [ ]:
# I'm a customer for 5 years, what will be the final price of the laptop?
# I want to buy an iPhone. Can you check stock?
messages = [
    {"role": "system", "content": "You are a helpful assistant for an online electronics store. As per the user requests try to answer the queries and make best use of tools available to achieve the goal"},
    {"role": "user", "content": "I'm a customer for 5 years, what will be the final price of the laptop?"}
]

# Calling LLM
response = ollama.chat(
    model="qwen3-vl:4b",
    messages=messages,
    tools=tools,
)

print(response)

model='qwen3-vl:4b' created_at='2026-02-08T07:36:50.2437792Z' done=True done_reason='stop' total_duration=13959587300 load_duration=2854120700 prompt_eval_count=310 prompt_eval_duration=114238400 eval_count=623 eval_duration=10856237800 message=Message(role='assistant', content='', thinking='Okay, let\'s see. The user wants to know the final price of a laptop after applying their loyalty discount because they\'ve been a customer for 5 years.\n\nFirst, I need to figure out the base price of the laptop. But wait, the user didn\'t mention the product name. Hmm, they just said "laptop," but maybe there are different models. Wait, the problem is that the function check_inventory requires the product name. But the user didn\'t specify which laptop. Maybe I should assume a standard laptop, but no, the tool needs the exact name.\n\nWait, the user is asking about a laptop, but the function check_inventory needs the product name. Since the user didn\'t specify a particular model, maybe there\'s 

In [8]:
response["message"]

Message(role='assistant', content='', thinking='Okay, let\'s see. The user wants to know the final price of a laptop after applying their loyalty discount because they\'ve been a customer for 5 years.\n\nFirst, I need to figure out the base price of the laptop. But wait, the user didn\'t mention the product name. Hmm, they just said "laptop," but maybe there are different models. Wait, the problem is that the function check_inventory requires the product name. But the user didn\'t specify which laptop. Maybe I should assume a standard laptop, but no, the tool needs the exact name.\n\nWait, the user is asking about a laptop, but the function check_inventory needs the product name. Since the user didn\'t specify a particular model, maybe there\'s a default or maybe the user expects me to get the base price. Wait, but in the problem statement, the tools available are check_inventory and calculate_loyalty_discount.\n\nSo, first, I need to check the inventory of the laptop. But the user has

In [9]:
# Check if the model asked to use any tools
tool_calls = response.message.get("tool_calls", [])

print("Tool calls:", tool_calls)
if tool_calls:
    for call in tool_calls:
        tool_name = call.function.name
        arguments = call.function.arguments
        
        # Get the actual python function
        function_to_call = available_functions.get(tool_name)

        # Run the function with given arguments
        result = function_to_call(**arguments)

        # Add the model's tool request to the conversation
        messages.append(response.message)  # Add the model's message to the conversation

        # Add the tool's result back to the conversation
        messages.append({
            "role": "tool", 
            "content": f"Result of {tool_name}: {result}"
        })

Tool calls: [ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))]


In [10]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant for an online electronics store.'},
 {'role': 'user',
  'content': "I'm a customer for 5 years, what will be the final price of the laptop?"},
 Message(role='assistant', content='', thinking='Okay, let\'s see. The user wants to know the final price of a laptop after applying their loyalty discount because they\'ve been a customer for 5 years.\n\nFirst, I need to figure out the base price of the laptop. But wait, the user didn\'t mention the product name. Hmm, they just said "laptop," but maybe there are different models. Wait, the problem is that the function check_inventory requires the product name. But the user didn\'t specify which laptop. Maybe I should assume a standard laptop, but no, the tool needs the exact name.\n\nWait, the user is asking about a laptop, but the function check_inventory needs the product name. Since the user didn\'t specify a particular model, maybe there\'s a default or maybe the user expects me 

In [12]:
# Ask the model to gennerate the final response after tool calls
final_response = ollama.chat(
    model="qwen3-vl:4b",
    messages=messages
)
print(final_response["message"]["content"])


In [14]:
final_response.__dict__

{'model': 'qwen3-vl:4b',
 'created_at': '2026-02-08T07:38:15.5338477Z',
 'done': True,
 'done_reason': 'stop',
 'total_duration': 19067545900,
 'load_duration': 121200200,
 'prompt_eval_count': 701,
 'prompt_eval_duration': 358990400,
 'eval_count': 1035,
 'eval_duration': 18524734900,
 'message': Message(role='assistant', content='', thinking='Okay, let me see. The user asked for the final price of their laptop after 5 years as a customer. I first needed to check the inventory for the laptop. The response from check_inventory shows a base price of $1200.\n\nNow, I need to apply the loyalty discount. The user has been a customer for 5 years. The function calculate_loyalty_discount requires the base_price and years_as_customer. \n\nWait, the user\'s years are 5. So, the discount percentage for 5 years. The problem statement might have the discount rate. Since the tools are part of the system, but the user didn\'t specify, but in the context of the problem, I need to assume the discount 